#### Study Assistant with Full Knowledge Base

##### What this notebook does

This is the working prototype for the Autonomous Agent Challenge (LAB M4.04).
It loads the **entire bootcamp knowledge base** (15 documents across 4 modules), chunks and embeds everything, and answers questions with document-specific citations.

**Pipeline:**
1. Load all 15 `.md` files from `bootcamp_knowledge_base/` (M1–M4)
2. Chunk every document with `RecursiveCharacterTextSplitter`
3. Embed all chunks in batches using OpenAI `text-embedding-3-small`
4. Store everything in Pinecone under namespace `study-assistant-prototype`
5. Embed a question → retrieve top-4 chunks → Claude answers from context only

**Knowledge base:**
| Module | Files |
|--------|-------|
| M1 Python Foundations | HomeGuard · Refactoring · APIs & Audio |
| M2 APIs & RAG | Prompt engineering · News summarizer · Chunking strategies · RAG pipeline |
| M3 LangChain | LangChain agents · Reranking |
| M4 Autonomous Agents | LangGraph · MCP · n8n · Media agent · Autonomous agent challenge |

---
**Pinecone index:** `study-assistant-prototype` (namespace: `study-assistant-prototype`)

**Embedding model:** `text-embedding-3-small` (OpenAI — vectors only)

**Answer model:** `claude-sonnet-4-5` (Anthropic — reads context, writes cited answer)

---
##### Cell 1: Install and Import

We need these libraries. Most are already installed from previous labs.

Note: we install BOTH `openai` (for embeddings) and `anthropic` (for answering).
They do different jobs in this pipeline.

In [15]:
# Install required libraries
# openai     = for text-embedding-3-small (embeddings only)
# anthropic  = for claude-sonnet-4-5 (answering questions)
# pinecone   = vector database
# langchain  = recursive text splitter for chunking
!pip install openai anthropic pinecone python-dotenv langchain tiktoken -q

In [16]:
# Cell 1: Imports and API key setup

import os
import time
import hashlib
import anthropic
from dotenv import load_dotenv
from openai import OpenAI
from pinecone import Pinecone, ServerlessSpec
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load API keys from .env file
load_dotenv()

# OpenAI client - used ONLY for creating embeddings
openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# Anthropic client - used for generating answers
anthropic_client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

# Pinecone client - vector database
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

# Confirm everything loaded
print("OpenAI client ready (embeddings):", openai_client is not None)
print("Anthropic client ready (answering):", anthropic_client is not None)
print("Pinecone client ready:", pc is not None)
print()
print("All clients loaded. Ready to build the prototype.")

OpenAI client ready (embeddings): True
Anthropic client ready (answering): True
Pinecone client ready: True

All clients loaded. Ready to build the prototype.


---
##### Cell 2: Load All Documents from the Knowledge Base

Instead of loading one file, we loop over every `.md` file in the knowledge base folder.
Each document keeps its filename and module folder as metadata so citations are specific.

The `documents` list holds every file as a dict with `text`, `source`, and `module` keys.

In [17]:
# Cell 2: Load all documents from the knowledge base

import os
from pathlib import Path

# Go up two levels from this notebook (Week 4/LAB4.04 -> bootcamp_env)
KB_DIR = Path(os.getcwd()) / "../../bootcamp_knowledge_base"
md_files = sorted(KB_DIR.glob("**/*.md"))

documents = []
for filepath in md_files:
    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()
    documents.append({
        "text": text,
        "source": filepath.name,
        "module": filepath.parent.name,
    })

print(f"Loaded {len(documents)} documents from knowledge base\n")
for doc in documents:
    print(f"  {doc['module']}/{doc['source']}  ({len(doc['text'])} chars)")

Loaded 15 documents from knowledge base

  M1_python_foundations/m1_01_homeguard.md  (1558 chars)
  M1_python_foundations/m1_02_refactoring.md  (2105 chars)
  M1_python_foundations/m1_05_07_api_and_audio.md  (1919 chars)
  M2_apis_and_rag/m2_01_prompt_engineering.md  (1815 chars)
  M2_apis_and_rag/m2_03_news_summarizer.md  (2011 chars)
  M2_apis_and_rag/m2_04_chunking_strategies.md  (2229 chars)
  M2_apis_and_rag/rag_pipeline_and_dynamic_prompting.md  (2750 chars)
  M3_langchain/m3_01_langchain_agents.md  (2743 chars)
  M3_langchain/m3_02_reranking.md  (2570 chars)
  M4_autonomous_agents/m4_01_langgraph.md  (3593 chars)
  M4_autonomous_agents/m4_02_mcp.md  (2919 chars)
  M4_autonomous_agents/m4_03_n8n.md  (3784 chars)
  M4_autonomous_agents/m4_04_autonomous_agent_challenge.md  (2472 chars)
  M4_autonomous_agents/project3_media_inclusivity_agent.md  (3152 chars)
  bootcamp_knowledge_base/README.md  (3212 chars)


---
##### Cell 3: Chunk All Documents

We loop over every document and split it into chunks.
Each chunk keeps the `source` filename and `module` folder as metadata
so the citation at the end of every answer is document-specific.

In [18]:
# Cell 3: Chunk all documents

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50
)

all_chunks = []  # list of dicts: {text, source, module}

for doc in documents:
    doc_chunks = splitter.split_text(doc["text"])
    for chunk_text in doc_chunks:
        all_chunks.append({
            "text":   chunk_text,
            "source": doc["source"],
            "module": doc["module"],
        })

print(f"Total chunks across all documents: {len(all_chunks)}\n")

# Show breakdown per document
from collections import Counter
counts = Counter(c["source"] for c in all_chunks)
for source, n in sorted(counts.items()):
    print(f"  {source}: {n} chunks")

Total chunks across all documents: 141

  README.md: 12 chunks
  m1_01_homeguard.md: 7 chunks
  m1_02_refactoring.md: 7 chunks
  m1_05_07_api_and_audio.md: 6 chunks
  m2_01_prompt_engineering.md: 6 chunks
  m2_03_news_summarizer.md: 8 chunks
  m2_04_chunking_strategies.md: 8 chunks
  m3_01_langchain_agents.md: 10 chunks
  m3_02_reranking.md: 9 chunks
  m4_01_langgraph.md: 13 chunks
  m4_02_mcp.md: 11 chunks
  m4_03_n8n.md: 13 chunks
  m4_04_autonomous_agent_challenge.md: 9 chunks
  project3_media_inclusivity_agent.md: 12 chunks
  rag_pipeline_and_dynamic_prompting.md: 10 chunks


---
##### Cell 4: Embed the Chunks and Store in Pinecone

We use OpenAI `text-embedding-3-small` to convert each chunk to a vector.
Then we store everything in a new Pinecone index called `study-assistant-prototype`.

**Important:** We use OpenAI for embeddings even though we use Anthropic for answering.
That is fine because embeddings and the LLM are completely separate steps.
The embedding model just converts text to numbers for Pinecone to search.
The LLM reads the retrieved text and writes the answer.
They never touch each other directly.

**Content hash IDs:** Each chunk gets an ID generated from its own text using `hashlib.md5`.
This means re-running this cell will not create duplicates - same text = same ID = Pinecone
overwrites it (that is what `upsert` means: update or insert).

In [19]:
# Cell 4: Embed all chunks and store in Pinecone

INDEX_NAME      = "study-assistant-prototype"
NAMESPACE       = "study-assistant-prototype"
EMBEDDING_MODEL = "text-embedding-3-small"
BATCH_SIZE      = 100   # embed 100 chunks per API call to stay within limits

# --- Step 1: Connect to or create the Pinecone index ---
existing_indexes = [idx.name for idx in pc.list_indexes()]
print(f"Existing Pinecone indexes: {existing_indexes}")

if INDEX_NAME in existing_indexes:
    active_index_name = INDEX_NAME
    print(f"Connected to existing index '{active_index_name}'")
else:
    try:
        print(f"Creating index '{INDEX_NAME}'...")
        pc.create_index(
            name=INDEX_NAME, dimension=1536, metric="cosine",
            spec=ServerlessSpec(cloud="aws", region="us-east-1")
        )
        print("Waiting 10 seconds for index to be ready...")
        time.sleep(10)
        active_index_name = INDEX_NAME
    except Exception as e:
        if "max serverless indexes" in str(e) or "FORBIDDEN" in str(e):
            active_index_name = existing_indexes[0]
            print(f"Index limit reached. Reusing '{active_index_name}' "
                  f"with namespace '{NAMESPACE}'")
        else:
            raise

index = pc.Index(active_index_name)

# --- Step 2: Embed all chunks in batches ---
print(f"\nEmbedding {len(all_chunks)} chunks in batches of {BATCH_SIZE}...")
all_texts      = [c["text"] for c in all_chunks]
all_embeddings = []

for i in range(0, len(all_texts), BATCH_SIZE):
    batch    = all_texts[i : i + BATCH_SIZE]
    response = openai_client.embeddings.create(model=EMBEDDING_MODEL, input=batch)
    all_embeddings.extend([item.embedding for item in response.data])
    print(f"  Embedded {min(i + BATCH_SIZE, len(all_texts))}/{len(all_texts)} chunks")

print(f"Each embedding: {len(all_embeddings[0])} dimensions")

# --- Step 3: Build vectors with metadata ---
vectors = []
for chunk_data, embedding in zip(all_chunks, all_embeddings):
    chunk_id = hashlib.md5(chunk_data["text"].encode()).hexdigest()
    vectors.append({
        "id":     chunk_id,
        "values": embedding,
        "metadata": {
            "text":   chunk_data["text"],
            "source": chunk_data["source"],
            "module": chunk_data["module"],
        }
    })

# --- Step 4: Upsert in batches ---
print(f"\nUploading {len(vectors)} vectors to '{active_index_name}' / '{NAMESPACE}'...")
for i in range(0, len(vectors), BATCH_SIZE):
    index.upsert(vectors=vectors[i : i + BATCH_SIZE], namespace=NAMESPACE)
    print(f"  Upserted {min(i + BATCH_SIZE, len(vectors))}/{len(vectors)}")

print("\nDone. Full knowledge base is in Pinecone.")

Existing Pinecone indexes: ['reranker-lab', 'trustworthy-ai-rag', 'media-inclusivity-index', 'n8n', 'study-assistant-prototype']
Connected to existing index 'study-assistant-prototype'

Embedding 141 chunks in batches of 100...
  Embedded 100/141 chunks
  Embedded 141/141 chunks
Each embedding: 1536 dimensions

Uploading 141 vectors to 'study-assistant-prototype' / 'study-assistant-prototype'...
  Upserted 100/141
  Upserted 141/141

Done. Full knowledge base is in Pinecone.


---
##### Cell 5: Ask a Question and Get a Cited Answer

Here is exactly what happens when this cell runs:

1. Your question is embedded using OpenAI (same model used for the chunks)
2. Pinecone finds the 4 most similar chunks using cosine similarity
3. Those 4 chunks are passed to Claude as context
4. Claude reads the context and writes a grounded answer
5. The answer is printed with the source file name

**Why Claude for answering?**
Claude is very good at following strict instructions. The key instruction here is:
`Only answer using the context provided below.`
Claude reliably stays within what was retrieved and says so when it cannot find something,
rather than filling gaps with general knowledge.

**Why OpenAI for embedding?**
The embedding model just converts text to numbers. It does not generate any answers.
We keep `text-embedding-3-small` because it is already used in previous labs
and matches what is already stored in Pinecone.

In [20]:
# Cell 5: Ask a question and get a cited answer

question = "What is the difference between cosine similarity and reranking?"

# --- Step 1: Embed the question ---
q_embed_response = openai_client.embeddings.create(
    model=EMBEDDING_MODEL, input=question
)
question_vector = q_embed_response.data[0].embedding

# --- Step 2: Search Pinecone ---
search_results = index.query(
    vector=question_vector, top_k=4,
    include_metadata=True, namespace=NAMESPACE
)
matches = search_results["matches"]
top_score = matches[0]["score"]

print(f"Retrieved {len(matches)} chunks | Top score: {top_score:.3f}\n")

# --- Step 3: Build context with source labels ---
context_parts = []
for m in matches:
    src  = m["metadata"].get("source", "unknown")
    text = m["metadata"]["text"]
    context_parts.append(f"[From {src}]\n{text}")
context = "\n\n---\n\n".join(context_parts)

# --- Step 4: Build prompt ---
sources_list = ", ".join(sorted({m["metadata"].get("source", "unknown") for m in matches}))
prompt = f"""You are a study assistant helping a student review their bootcamp notes.
Answer using ONLY the context below. Do not use any outside knowledge.
If the answer is not in the context, say: "I could not find this in your course notes."
Be clear and structured. Use bullet points where helpful.
End with: Source: [exact filename(s) where you found the answer]

Context:
{context}

Question: {question}

Answer:"""

# --- Step 5: Send to Claude ---
response = anthropic_client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=1024,
    messages=[{"role": "user", "content": prompt}]
)
answer = response.content[0].text

print("=" * 60)
print("QUESTION:", question)
print()
print("ANSWER:")
print(answer)
print("=" * 60)

Retrieved 4 chunks | Top score: 0.744

QUESTION: What is the difference between cosine similarity and reranking?

ANSWER:
# Difference Between Cosine Similarity and Reranking

## Cosine Similarity
- Measures **general semantic closeness**
- Scores tend to **cluster tightly together** (e.g., 0.84 to 0.89)
- Makes it hard to distinguish truly relevant results from loosely related ones
- A chunk that might actually be the best answer could be ranked as low as 7th

## Reranking
- **Trained specifically to score query-document relevance**
- Scores **spread meaningfully** (e.g., 0.44 to 0.85)
- Better at identifying which chunks are truly relevant
- Can correctly promote chunks from lower positions (e.g., from 7th place to 1st place)

## Performance Impact
- Baseline retrieval (cosine only): 1 out of 4 test queries answered correctly
- With reranking: 2 out of 4 test queries answered correctly

**Source:** m3_02_reranking.md


---
##### Cell 6: Reusable Function + Two More Test Questions

We wrap the full pipeline in a clean function called `ask_study_assistant()`.
This is the pattern that will be used when building the full agent.

Notice the function takes both `openai_client` (for embedding) and `anthropic_client`
(for answering) as parameters. Keeping them separate makes it easy to swap either one later.

In [21]:
# Cell 6: Reusable function and two more test questions

def ask_study_assistant(question, index, openai_client, anthropic_client,
                        namespace=NAMESPACE, top_k=4):
    # Embed question
    q_vec = openai_client.embeddings.create(
        model=EMBEDDING_MODEL, input=question
    ).data[0].embedding

    # Search Pinecone
    results = index.query(
        vector=q_vec, top_k=top_k,
        include_metadata=True, namespace=namespace
    )
    matches = results["matches"]

    # Build context with source labels so Claude knows which doc each chunk is from
    context_parts = []
    for m in matches:
        src  = m["metadata"].get("source", "unknown")
        text = m["metadata"]["text"]
        context_parts.append(f"[From {src}]\n{text}")
    context = "\n\n---\n\n".join(context_parts)

    prompt = f"""You are a study assistant helping a student review their bootcamp notes.
Answer using ONLY the context below. Do not use any outside knowledge.
If the answer is not in the context, say: "I could not find this in your course notes."
Be clear and structured. Use bullet points where helpful.
End with: Source: [exact filename(s) where you found the answer]

Context:
{context}

Question: {question}

Answer:"""

    response = anthropic_client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text


# --- Test across multiple modules ---
test_questions = [
    "What is LangGraph and how is it different from a regular LangChain agent?",
    "What chunking strategies did we compare in the bootcamp?",
    "How do I set up a Flask web server?",   # should trigger the fallback
]

for i, q in enumerate(test_questions, 1):
    print(f"TEST {i}: {q}")
    print()
    print(ask_study_assistant(q, index, openai_client, anthropic_client))
    print("\n" + "-" * 60 + "\n")

TEST 1: What is LangGraph and how is it different from a regular LangChain agent?

# LangGraph vs LangChain Agent

## What is LangGraph?
LangGraph is a framework where **YOU define exactly what happens in what order**, making it predictable and auditable.

## Key Differences

**LangChain Agent:**
- The LLM decides what to do next
- Flexible but unpredictable
- Works like a detective - improvises based on the situation

**LangGraph:**
- You control the exact flow and order of operations
- Predictable and auditable
- Works like a hospital admissions process - follows a fixed protocol every time

## Main Characteristic
LangGraph is **deterministic** - the same input always follows the same path. This makes it:
- Auditable
- Predictable
- Critical for business processes that need consistency

## Tradeoff
The tradeoff is **less flexibility** - you have to anticipate every possible path upfront, unlike LangChain agents where the LLM can improvise.

---

**Source:** m4_01_langgraph.md

------

---
##### Cell 7: Gradio Interface

A simple UI that wraps `ask_study_assistant()`.
It reuses the `index`, `openai_client`, and `anthropic_client` already loaded in this kernel — no re-embedding needed.

Run this cell and click the public link to share it.

In [22]:
# Cell 7: Gradio interface — runs inline, reuses all kernel variables

import gradio as gr

def ask(question: str):
    if not question.strip():
        return "", ""

    answer = ask_study_assistant(
        question, index, openai_client, anthropic_client,
        namespace=NAMESPACE, top_k=4
    )

    # Pull the raw matches again for the source panel
    q_vec = openai_client.embeddings.create(
        model=EMBEDDING_MODEL, input=question
    ).data[0].embedding
    matches = index.query(
        vector=q_vec, top_k=4, include_metadata=True, namespace=NAMESPACE
    )["matches"]

    top_score = matches[0]["score"] if matches else 0
    source_lines = []
    for i, m in enumerate(matches, 1):
        src     = m["metadata"].get("source", "unknown")
        module  = m["metadata"].get("module", "")
        score   = m["score"]
        preview = m["metadata"]["text"][:150].replace("\n", " ")
        source_lines.append(
            f"**Chunk {i}** — `{module}/{src}` (score: {score:.3f})\n> {preview}…"
        )

    sources_md = f"**Top score:** {top_score:.3f}\n\n" + "\n\n".join(source_lines)
    return answer, sources_md


with gr.Blocks(title="Study Assistant", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        "# Study Assistant\n"
        "Searches your full bootcamp knowledge base — **OpenAI embeddings · Claude answers · Pinecone**"
    )

    with gr.Row():
        with gr.Column(scale=3):
            question_box = gr.Textbox(
                label="Your question",
                placeholder="e.g. What is LangGraph and how does it differ from a regular agent?",
                lines=2,
            )
            ask_btn = gr.Button("Ask", variant="primary")

        with gr.Column(scale=1):
            gr.Markdown(
                "**Knowledge base** — 15 docs\n\n"
                "**M1** Python Foundations (3)\n"
                "HomeGuard · Refactoring · APIs & Audio\n\n"
                "**M2** APIs & RAG (4)\n"
                "Prompt engineering · News summarizer · Chunking · RAG pipeline\n\n"
                "**M3** LangChain (2)\n"
                "LangChain agents · Reranking\n\n"
                "**M4** Autonomous Agents (5)\n"
                "LangGraph · MCP · n8n · Media agent · Agent challenge\n\n"
                "---\n"
                "**Embeddings:** `text-embedding-3-small`\n\n"
                "**Answers:** `claude-sonnet-4-5`"
            )

    answer_box = gr.Markdown(label="Answer")

    with gr.Accordion("Retrieved source chunks", open=False):
        sources_box = gr.Markdown()

    ask_btn.click(fn=ask, inputs=question_box, outputs=[answer_box, sources_box])
    question_box.submit(fn=ask, inputs=question_box, outputs=[answer_box, sources_box])

    gr.Examples(
        examples=[
            "What is LangGraph and how is it different from a regular LangChain agent?",
            "What chunking strategies did we compare in the bootcamp?",
            "Why do cosine similarity scores cluster together?",
            "What is MCP and what problem does it solve?",
            "How do I set up a Flask web server?",
        ],
        inputs=question_box,
    )

demo.launch(share=True)

/var/folders/gh/r4_2cb497nl1c31dzl76npj80000gp/T/ipykernel_52926/489728687.py:37: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Study Assistant", theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://bd87b5bd5dc3ca9834.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
#### Prototype Summary

##### What we built

A working RAG pipeline across the full bootcamp knowledge base.

| Step | What happened |
|---|---|
| Load | Looped over all 15 `.md` files in `bootcamp_knowledge_base` using `Path.glob` |
| Chunk | Split every document using RecursiveCharacterTextSplitter (400 chars, 50 overlap) |
| Embed | Batched all chunks in groups of 100, converted to 1536-dim vectors (OpenAI `text-embedding-3-small`) |
| Store | Uploaded all vectors to Pinecone index `study-assistant-prototype` with namespace and source metadata |
| Query | Embedded the question with OpenAI, searched Pinecone across the full knowledge base |
| Answer | Retrieved chunks labelled with `[From filename]`, passed to Claude, got a document-specific cited answer |

##### Why two different API providers?

- **OpenAI** (`text-embedding-3-small`): used only for converting text to vectors. Must stay consistent with what is already stored in Pinecone.
- **Anthropic** (`claude-sonnet-4-5`): used only for generating the answer. Claude reliably follows the `only answer from context` instruction.

##### Key things learned from building this

- Batching embeddings in groups of 100 is required - sending all chunks at once fails above a certain size
- Namespaces in Pinecone isolate this project's vectors from others in the same index (important on the free tier)
- Adding `[From filename]` labels to each chunk in the context means Claude can cite the exact source file
- Including a deliberate fallback test (Flask question not in KB) verifies the not-found instruction works
- `time.sleep(10)` after creating a Pinecone index is required - it is not immediately usable
- Anthropic response is at `response.content[0].text`, not `response.choices[0].message.content` like OpenAI

#### What the full autonomous agent still needs

- LangGraph workflow: plan node, research node (with ReAct retry loop), synthesize node
- Session memory so follow-up questions work within one conversation
- Simple text input interface so the student does not edit the question variable in code (added a Gradio in Step 7)
- Cohere reranking on top of vector search for better retrieval quality (already learned in Lab 3.02)
